<a href="https://colab.research.google.com/github/23070075-creator/GenAI-Pacticals/blob/main/Gen_Ai_pra_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers datasets accelerate

import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score

model_name = "distilbert/distilbert-base-uncased"

data = {
    "text": [
        "The team won the cricket championship",
        "The player scored a goal in football",
        "The coach announced the final squad",
        "The team won the world cup",
        "The government announced a new policy",
        "Parliament discussed the national budget",
        "The election campaign started today",
        "The minister addressed the assembly",
        "Artificial intelligence is changing technology",
        "The company developed a new computer chip",
        "Python is widely used in machine learning",
        "Scientists introduced a new software system"
    ],
    "label": [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]
}

dataset = Dataset.from_dict(data).train_test_split(
    test_size=0.25, seed=42
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

args = TrainingArguments(
    output_dir="./news_results",
    eval_strategy="epoch",
    save_strategy="no",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

trainer.train()
print(trainer.evaluate())

def classify_news(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits

    labels = ["Sports", "Politics", "Technology"]
    return labels[logits.argmax(dim=1).item()]

print(classify_news(
    "The company launched a new artificial intelligence chip"
))